# Visualization and Reporting

This notebook aggregates results from the LLM Judge and Traditional Analysis workflows to generate comprehensive visualizations and reports.

## Objectives
- Combine analysis results
- Generate comparative plots (Violin plots, Correlation matrices)
- Create summary tables and rankings
- Save all artifacts

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import seaborn as sns
import matplotlib.pyplot as plt
import sys

# Add current directory to path
sys.path.append(os.getcwd())

from config import *

# Visualization Settings
sns.set_theme(style="whitegrid", font_scale=FONT_SCALE)
plt.rcParams['figure.figsize'] = FIG_SIZE_SINGLE

In [ ]:
def load_and_merge_data():
    """Load and merge data from analysis directories"""
    print("Loading analysis results...")
    
    # Load LLM Judge Results
    llm_files = glob.glob(os.path.join(LLM_JUDGE_DIR, "*.csv"))
    if not llm_files:
        print(f"Warning: No files found in {LLM_JUDGE_DIR}")
        llm_df = pd.DataFrame()
    else:
        llm_df = pd.concat((pd.read_csv(f) for f in llm_files), ignore_index=True)
        print(f"Loaded {len(llm_df)} rows from LLM Judge analysis")
    
    # Load Traditional Analysis Results
    trad_files = glob.glob(os.path.join(TRADITIONAL_DIR, "*.csv"))
    if not trad_files:
        print(f"Warning: No files found in {TRADITIONAL_DIR}")
        trad_df = pd.DataFrame()
    else:
        trad_df = pd.concat((pd.read_csv(f) for f in trad_files), ignore_index=True)
        print(f"Loaded {len(trad_df)} rows from Traditional analysis")
        
    if llm_df.empty or trad_df.empty:
        print("Error: Missing data from one or both analysis pipelines.")
        return pd.DataFrame()
    
    # Merge Logic
    # We assume 'model', 'strategy', 'reasoning' are sufficient unique identifiers
    # Note: 'option_selected' or other input cols could also be used
    merge_cols = ['model', 'strategy', 'reasoning']
    
    # Find common columns to verify merge keys
    common_cols = list(set(llm_df.columns) & set(trad_df.columns))
    print(f"Common columns: {common_cols}")
    
    # Drop duplicates columns in trad_df except merge keys to avoid _x, _y
    cols_to_use = merge_cols + [c for c in trad_df.columns if c not in common_cols]
    trad_df_clean = trad_df[cols_to_use]
    
    merged_df = pd.merge(llm_df, trad_df_clean, on=merge_cols, how='inner')
    print(f"Merged DataFrame shape: {merged_df.shape}")
    
    # Clean numeric columns for analysis
    score_cols = [c for c in merged_df.columns if c.endswith('_score')]
    for col in score_cols:
        merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')
        
    return merged_df

results_df = load_and_merge_data()

In [ ]:
# Determine metric columns
if not results_df.empty:
    judge_cols = [col for col in results_df.columns if col.endswith('_score')]
    traditional_cols = [
        'reasoning_word_count', 'vocabulary_diversity', 
        'total_factors_mentioned', 'has_comparison'
    ]
    # Filter to existing columns
    traditional_cols = [c for c in traditional_cols if c in results_df.columns]
    
    print(f"Judge Metrics: {judge_cols}")
    print(f"Traditional Metrics: {traditional_cols}")

In [ ]:
# Visualization 1: Violin Plots for Traditional Metrics
if not results_df.empty and traditional_cols:
    print("Generating Violin Plots...")
    
    # Setup grid
    n_metrics = len(traditional_cols)
    rows = (n_metrics + 1) // 2
    fig, axes = plt.subplots(rows, 2, figsize=(16, rows*6))
    axes = axes.flatten()
    
    for i, metric in enumerate(traditional_cols):
        sns.violinplot(
            data=results_df, 
            x='model', 
            y=metric, 
            hue='strategy', 
            ax=axes[i], 
            split=False
        )
        axes[i].set_title(f'Distribution of {metric} by Model', fontsize=14)
        axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45)
        axes[i].legend(loc='upper right')
        
    # Hide empty subplots
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
        
    plt.tight_layout()
    save_path = os.path.join(PLOTS_DIR, 'traditional_metrics_violin.png')
    plt.savefig(save_path, dpi=DPI)
    print(f"Saved violin plots to {save_path}")
    plt.show()

In [ ]:
# Visualization 2: Correlation Matrix
if not results_df.empty:
    print("Generating Correlation Matrix...")
    
    # Select all numeric columns of interest
    corr_cols = judge_cols + traditional_cols
    corr_df = results_df[corr_cols].dropna()
    
    if not corr_df.empty:
        plt.figure(figsize=(12, 10))
        correlation = corr_df.corr()
        
        sns.heatmap(
            correlation, 
            annot=True, 
            fmt='.2f', 
            cmap='coolwarm', 
            center=0,
            square=True,
            cbar_kws={"shrink": .8}
        )
        
        plt.title('Correlation between Judge Scores and Traditional Metrics', fontsize=16)
        plt.tight_layout()
        save_path = os.path.join(PLOTS_DIR, 'correlation_matrix.png')
        plt.savefig(save_path, dpi=DPI)
        print(f"Saved correlation matrix to {save_path}")
        plt.show()
    else:
        print("Not enough data for correlation analysis")

In [ ]:
# Visualization 3: Model Comparison Table
if not results_df.empty:
    print("Generating Model Ranking...")
    
    # Calculate Overall Quality Score (Average of all LLM judge scores)
    if judge_cols:
        results_df['overall_judge_score'] = results_df[judge_cols].mean(axis=1)
        
        # Aggregation by Model
        ranking = results_df.groupby('model').agg({
            'overall_judge_score': 'mean',
            'reasoning_word_count': 'mean',
            'vocabulary_diversity': 'mean'
        }).round(3)
        
        # Sort by overall score
        ranking = ranking.sort_values('overall_judge_score', ascending=False)
        
        print("\nModel Ranking (by LLM Judge Score):")
        display(ranking)
        
        # Save ranking
        csv_path = os.path.join(OUTPUT_DIR, 'model_ranking.csv')
        ranking.to_csv(csv_path)
        print(f"Saved ranking to {csv_path}")
        
        # Plot Ranking
        plt.figure(figsize=(10, 6))
        sns.barplot(x=ranking.index, y='overall_judge_score', data=ranking, palette='viridis')
        plt.title('Overall Model Performance (Judge Score)', fontsize=16)
        plt.ylabel('Average Score (0-1)')
        plt.xticks(rotation=45)
        plt.tight_layout()
        
        save_path = os.path.join(PLOTS_DIR, 'model_ranking_bar.png')
        plt.savefig(save_path, dpi=DPI)
        print(f"Saved ranking plot to {save_path}")
        plt.show()